# Component 3 DDK demonstration (inference only)

This notebook does **not** import or execute the training notebook. It loads the already-trained Component 3 model bundle, reuses the shared feature extractor, and runs inference on:

- one explicitly selected Parkinson's-disease participant from the labeled research dataset;
- the two personal recordings, whose clinical label is unverified.

This is an illustrative demonstration, not a new accuracy evaluation or a medical diagnosis. No model, dataset, evaluation artifact, or recording is modified.

In [ ]:
# %pip install imageio-ffmpeg

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the repository root importable when this notebook is run from its folder.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parents[1]
elif (REPO_ROOT / "parkinson-speech-ai").is_dir():
    REPO_ROOT = REPO_ROOT / "parkinson-speech-ai"
if not (REPO_ROOT / "component3_ddk" / "features.py").is_file():
    raise FileNotFoundError(f"Could not locate repository root from {Path.cwd()}")
sys.path.insert(0, str(REPO_ROOT))

from component3_ddk.features import FEATURE_NAMES, extract_ddk_features

MODEL_PATH = REPO_ROOT / "models" / "component3_ddk_ddk_rf.joblib"
DATASET_ROOT = REPO_ROOT / "data" / "Italian Parkinson_s Voice and speech"
PERSONAL_ROOT = REPO_ROOT / "data" / "personal_ddk"

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Saved model bundle not found: {MODEL_PATH}")
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_ROOT}")
print("Inference-only demo; no training or artifact regeneration will be performed.")
print("Model:", MODEL_PATH)
print("Dataset:", DATASET_ROOT)
print("Personal recordings:", PERSONAL_ROOT)


In [ ]:
bundle = joblib.load(MODEL_PATH)
if "model" not in bundle or "feature_names" not in bundle:
    raise KeyError("Model bundle must contain 'model' and 'feature_names'.")
model = bundle["model"]
feature_names = list(bundle["feature_names"])
if feature_names != list(FEATURE_NAMES):
    raise ValueError("Saved model feature order does not match features.py.")
if not hasattr(model, "classes_") or not hasattr(model, "predict_proba"):
    raise AttributeError("Saved model must expose classes_ and predict_proba().")

def pd_probability(probabilities):
    for index, value in enumerate(model.classes_):
        if str(value).strip().lower() == "pd" or value == 1:
            return float(probabilities[index])
    raise ValueError(f"Could not identify PD class in {model.classes_!r}")

def predict_file(path, source, known_label):
    features = extract_ddk_features(str(path))
    if features is None:
        raise ValueError(f"Too few usable onsets detected in {path}")
    frame = pd.DataFrame([{name: features[name] for name in feature_names}], columns=feature_names)
    if not np.isfinite(frame[feature_names].to_numpy()).all():
        raise ValueError(f"Non-finite features extracted from {path}")
    probabilities = model.predict_proba(frame[feature_names])[0]
    predicted_value = model.predict(frame[feature_names])[0]
    predicted_class = "PD" if str(predicted_value).strip().lower() == "pd" or predicted_value == 1 else "HC"
    stem = path.stem.lower()
    task = "D1/pa" if stem.startswith("d1") or stem.endswith("-pa") else ("D2/ta" if stem.startswith("d2") or stem.endswith("-ta") else "unknown")
    return {
        "source": source,
        "known_label": known_label,
        "participant": path.parent.name,
        "filename": path.name,
        "task": task,
        "predicted_class": predicted_class,
        "pd_probability": pd_probability(probabilities),
    }

# Explicit, reproducible labeled PD example. Do not select by prediction.
pd_participant = DATASET_ROOT / "28 People with Parkinson's disease" / "1-5" / "Domenico C"
pd_paths = [pd_participant / name for name in ("D1cdaopmoe67M2605161905.wav", "D2cdaopmoe67M2605161905.wav")]
for path in pd_paths:
    if not path.is_file():
        raise FileNotFoundError(f"Expected labeled PD recording not found: {path}")

personal_paths = [PERSONAL_ROOT / "DDK001-pa.wav", PERSONAL_ROOT / "DDK001-ta.wav"]
for path in personal_paths:
    if not path.is_file():
        raise FileNotFoundError(f"Expected personal recording not found: {path}")

rows = [predict_file(path, "Labeled research dataset", "PD") for path in pd_paths]
rows.extend(predict_file(path, "Personal recording", "Unverified" ) for path in personal_paths)
demo_df = pd.DataFrame(rows)
display(demo_df[["source", "known_label", "participant", "filename", "task", "predicted_class", "pd_probability"]])


In [ ]:
plot_df = demo_df.copy()
plot_df["display_name"] = plot_df["source"].str.replace("Labeled research dataset", "PD dataset", regex=False).str.replace("Personal recording", "Personal", regex=False) + "\n" + plot_df["task"]
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(plot_df["display_name"], plot_df["pd_probability"], color=["#C0504D", "#C0504D", "#4472C4", "#4472C4"])
ax.set_ylim(0, 1)
ax.set_ylabel("Predicted PD probability")
ax.set_title("Component 3 DDK inference demonstration")
ax.axhline(0.5, color="#777777", linestyle="--", linewidth=1, label="0.5 reference threshold")
for bar, probability in zip(bars, plot_df["pd_probability"]):
    ax.text(bar.get_x() + bar.get_width() / 2, probability + 0.02, f"{probability:.1%}", ha="center")
ax.legend()
plt.tight_layout()
plt.show()

print("The PD dataset rows have a known research label: PD.")
print("The personal rows are unverified model outputs, not a diagnosis.")
print("This demonstration does not measure generalization accuracy.")
